# Official UniverSeg implementation check

This notebook is a separate reference check against the public UniverSeg implementation. It is not the training pipeline for the from-scratch model in this repository.

**Requirements:** an internet connection to clone the upstream repository, a compatible PyTorch installation, and the optional notebook dependencies in `requirements-optional.txt`.

**Data note:** the notebook downloads/uses the upstream example WBC dataset. Review the upstream dataset terms before redistributing any downloaded data.


In [ ]:
%pip install -q einops pydantic nibabel
from pathlib import Path
import subprocess
import sys

upstream_dir = Path("third_party") / "UniverSeg"
if not upstream_dir.exists():
    upstream_dir.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/JJGO/UniverSeg", str(upstream_dir)],
        check=True,
    )
sys.path.insert(0, str(upstream_dir.resolve()))


In [ ]:
import torch
from universeg import universeg

device = "cuda" if torch.cuda.is_available() else "cpu"
model = universeg(pretrained=True).to(device)
model.eval()
print(device)

In [ ]:
from example_data.wbc import WBCDataset

support_ds = WBCDataset(dataset="JTSC", split="support", label="cytoplasm")
test_ds = WBCDataset(dataset="JTSC", split="test", label="cytoplasm")

S = 16
support_images = torch.stack([support_ds[i][0] for i in range(S)])[None].to(device)
support_labels = torch.stack([support_ds[i][1] for i in range(S)])[None].to(device)

target_image, target_label = test_ds[0]
target_image = target_image[None].to(device)

with torch.no_grad():
    pred = model(target_image, support_images, support_labels)

print(pred.shape)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 3, figsize=(9,3))
ax[0].imshow(target_image[0,0].cpu(), cmap='gray'); ax[0].set_title("target image")
ax[1].imshow(target_label[0], cmap='gray'); ax[1].set_title("ground truth")
ax[2].imshow(torch.sigmoid(pred[0,0]).cpu() > 0.5, cmap='gray'); ax[2].set_title("prediction")
for a in ax: a.axis('off')
plt.show()

In [ ]:
def dice_score(pred, label, eps=1e-7):
    pred = (torch.sigmoid(pred) > 0.5).float()
    label = label.float()
    intersection = (pred * label).sum()
    return (2 * intersection + eps) / (pred.sum() + label.sum() + eps)

dice_scores = []
with torch.no_grad():
    for i in range(len(test_ds)):
        target_image, target_label = test_ds[i]
        target_image = target_image[None].to(device)
        target_label = target_label[None].to(device)
        pred = model(target_image, support_images, support_labels)
        dice_scores.append(dice_score(pred, target_label).item())

import numpy as np
print(f"Mean Dice over {len(dice_scores)} test images: {np.mean(dice_scores):.4f}")